# 03 — Linear Regression with BIC / 使用BIC的线性回归

**Chapter 15 — File 3 of 3**

## Summary

We implement the Bayesian Information Criterion (BIC) formula, which is an alternative to AIC. BIC applies a stronger penalty for model complexity (k*log(n) instead of 2k), making it prefer simpler models.

我们实现贝叶斯信息准则（BIC）公式，这是AIC的替代品。BIC对模型复杂性施加更强的惩罚（k*log(n)而不是2k），使其更喜欢简单模型。

**Formula:**

$$BIC = n \log(MSE) + k \log(n)$$

where $n$ is sample size and $k$ is number of parameters

## Step 1 — Calculate and Compare AIC vs BIC / 计算并比较AIC vs BIC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures

# Generate data / 生成数据
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = 2.5 * X.ravel() + np.random.normal(0, 2, 100)
n = len(y)

# Compare AIC and BIC for different polynomial degrees / 比较不同多项式阶数的AIC和BIC
degrees = range(1, 8)
aic_vals = []
bic_vals = []
mse_vals = []

print(f'AIC vs BIC Comparison / AIC vs BIC比较')
print(f'=' * 90)
print(f'{"Degree":>8s} {"k":>5s} {"MSE":>12s} {"AIC":>12s} {"BIC":>12s} {"Best by":>10s}')
print('-' * 90)

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree)
    X_poly = poly_features.fit_transform(X)
    
    model = LinearRegression()
    model.fit(X_poly, y)
    y_pred = model.predict(X_poly)
    
    mse = mean_squared_error(y, y_pred)
    k = X_poly.shape[1]
    
    # AIC = n*log(MSE) + 2k
    aic = n * np.log(mse) + 2 * k
    
    # BIC = n*log(MSE) + k*log(n)
    bic = n * np.log(mse) + k * np.log(n)
    
    mse_vals.append(mse)
    aic_vals.append(aic)
    bic_vals.append(bic)
    
    best = 'AIC' if aic < bic else 'BIC'
    print(f'{degree:8d} {k:5d} {mse:12.6f} {aic:12.4f} {bic:12.4f} {best:>10s}')

best_aic_degree = degrees[np.argmin(aic_vals)]
best_bic_degree = degrees[np.argmin(bic_vals)]
print(f'\nBest by AIC: Degree {best_aic_degree}')
print(f'Best by BIC: Degree {best_bic_degree}')

## Step 2 — Visualize Comparison / 可视化比较

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: AIC and BIC together / 绘图1：AIC和BIC一起
axes[0].plot(degrees, aic_vals, 'b-o', linewidth=2, markersize=8, label='AIC')
axes[0].plot(degrees, bic_vals, 'r-s', linewidth=2, markersize=8, label='BIC')
axes[0].axvline(best_aic_degree, color='b', linestyle='--', alpha=0.5, label=f'AIC best: {best_aic_degree}')
axes[0].axvline(best_bic_degree, color='r', linestyle='--', alpha=0.5, label=f'BIC best: {best_bic_degree}')
axes[0].set_xlabel('Polynomial Degree / 多项式阶数')
axes[0].set_ylabel('Criterion Value / 准则值')
axes[0].set_title('AIC vs BIC: Model Selection')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Difference between AIC and BIC / 绘图2：AIC和BIC之间的差异
diff = np.array(aic_vals) - np.array(bic_vals)
axes[1].plot(degrees, diff, 'g-^', linewidth=2, markersize=8)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Polynomial Degree / 多项式阶数')
axes[1].set_ylabel('AIC - BIC')
axes[1].set_title('BIC Penalizes Complexity More (AIC - BIC > 0 means BIC favors simpler model)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3 — Analyze Penalty Differences / 分析惩罚差异

In [ ]:
print(f'\nPenalty Comparison / 惩罚比较')
print(f'=' * 60)
print(f'For the same k parameters:')
print(f'\nAIC penalty: 2k')
print(f'BIC penalty: k*log(n) = k*log({n}) = k*{np.log(n):.4f}')
print(f'\nSince log({n}) ≈ {np.log(n):.4f} > 2, BIC penalizes complexity more')
print(f'\nExample: k=5')
print(f'  AIC penalty: 2*5 = 10')
print(f'  BIC penalty: 5*log({n}) = 5*{np.log(n):.4f} = {5*np.log(n):.4f}')
print(f'  BIC penalty is {5*np.log(n)/10:.2f}x AIC penalty')

## Learning Notes / 学习笔记

- **Concept**: BIC applies a stronger penalty than AIC, especially for large datasets. It's derived from Bayesian model comparison and has theoretical justification for model selection.

- **ML Application**: Choose AIC for prediction-focused tasks, BIC for inference and true model discovery. BIC is often preferred in scientific applications where finding the true model is important.

## Chapter 15 Complete / 第15章完成

This chapter covered information criteria (AIC and BIC) for balancing model fit and complexity.